# 05 - MCDA Analysis

Demonstrating TOPSIS and ProjectRanker for multi-criteria project prioritization.

## Objectives
- Use ProjectRanker and TOPSIS from src.mcda
- Show weight sensitivity analysis
- Visualize ranking results with different weight profiles

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path('..').resolve()))

import pandas as pd
from src.data import FeatureEngineer
from src.mcda import ProjectRanker

## 1. Load Data and Create Features

In [ ]:
df = pd.read_csv('../data/raw/sample_projects.csv')
fe = FeatureEngineer()
df = fe.create_features(df)

# Add synthetic risk_score and sentiment_score if not present (from ML/LLM)
if 'risk_score' not in df.columns:
    df['risk_score'] = 1 - (df['completion_rate'] / 100) if 'completion_rate' in df.columns else 0.5
if 'sentiment_score' not in df.columns:
    df['sentiment_score'] = 0.0

if 'project_id' not in df.columns:
    df['project_id'] = df['project_name'] if 'project_name' in df.columns else range(len(df))

print(f'Loaded {len(df)} projects')

## 2. Run MCDA Ranking

In [ ]:
ranker = ProjectRanker()
rankings = ranker.rank(df)

print(rankings[['project_name', 'mcda_score', 'rank', 'risk_level']].head(10))

## 3. Custom Weights

In [ ]:
custom_criteria = {
    'ml_risk_score': {'weight': 0.50, 'type': 'cost'},
    'llm_sentiment_score': {'weight': 0.20, 'type': 'benefit'},
    'schedule_performance_index': {'weight': 0.15, 'type': 'benefit'},
    'cost_performance_index': {'weight': 0.10, 'type': 'benefit'},
    'team_stability': {'weight': 0.05, 'type': 'benefit'},
}

ranker_quant = ProjectRanker(criteria=custom_criteria)
rankings_quant = ranker_quant.rank(df)
print('Quantitative-weighted top 5:')
print(rankings_quant[['project_name', 'mcda_score', 'rank']].head())

## 4. Sensitivity Analysis

In [ ]:
sensitivity = ranker.sensitivity_analysis(df, weight_variation=0.10)
print('Sensitivity by criterion:')
for crit, data in sensitivity['criteria_sensitivity'].items():
    print(f'  {crit}: avg rank change = {data["avg_rank_change"]:.2f}')

## 5. Visualize Rankings

In [ ]:
from src.visualization.risk_charts import RiskCharts

fig = RiskCharts.risk_score_bar(rankings, top_n=10)
fig.show()